[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eirasf/GCED-AA3/blob/main/en/lab1/lab1.ipynb)

# Practical 1:  Dimensionality reduction

## The problem of the curse of dimensionality

One of the main problems in machine learning is what is known as the curse of dimensionality. This problem arises from the very moment one tries to improve an approximation simply by using more variables. Perhaps it could be done, but most likely its effect will be counterproductive. This is where the aforementioned curse comes into play since, **as the number of features or dimensions increases, the amount of data needed
to obtain an accurate generalization increases exponentially.** However, before tackling the problem, it is necessary to know where it comes from.

![Behavior of the performance when the number of variables is increased](./Images/Perfomance_Dimension_plot.png)


### The dimensionality of problems

Rarely does one think about the impact that a given variable will have on an optimization process. Take, for example, a case in which there are 5 observations for a given variable **X**, and those observations are uniformly distributed in the space. In that way each of the observations will try to represent $\frac{1}{5}$ of the aforementioned space.

![](./Images/Sampling_Examples.png)

When a new variable **Y** is added, moving, therefore, to a two-dimensional space. In order to maintain the same distance between the samples, the same rate of space representativeness, the number of samples should be increased to 25. With a third one, 125 samples would be needed to explore the space under the same conditions, etc.

Therefore, the present problem grows exponentially the more dimensions we have.


### The curse

What happens in a real problem? Well, usually the number of samples cannot be increased to maintain the representativeness of the sample points and their equidistance. That is why, if we add a new feature but we do not provide enough points, the result would be a more complex model but with a degraded performance.

The reason for that claim can be clearly seen with the following example, in which the goal is to classify between cat and dog images. If only one dimension is taken into account, the examples are uniformly distributed.

![](./Images/Doom_1.png)

In this case there are 10 samples covering the whole space. But, if one more dimension is added, that distribution becomes something like the following figure.

![](./Images/Doom_2.png)

This situation could make one think that a new dimension would make dividing the space even easier. For example, adding a third dimension to the previous problem yields something like:

![](./Images/Doom_3.png)

Where the result is linearly separable as can be seen in the following image.

![](./Images/Doom_4.png)

The erroneous conclusion we could draw is that the more dimensionality increases, the easier the separation based on the features will be. Note how the distribution of the data has changed: while in one dimension there are 2 samples per interval of the five samples mentioned before, in three-dimensional space it barely reaches 0.08 samples per interval (10/125). Therefore, it is harder for counterexamples to be found on the same side of the classifier. The problem arises when we project that data to a lower dimensional space, as happens when any artificial neural network is applied to create a classifier. In that situation the result would be similar to the following figure:

![](./Images/Doom_5.png)

As can be seen in the following figure, the classifier has been overfitted and, therefore, the result is not as good for new instances as it could be. For example, see the following figure where a simple linear classifier has been applied on fewer dimensions

![](./Images/Doom_6.png)


### How to avoid the curse?

There is no fixed rule that defines how many features should be used in a regression/classification problem. The number will depend on the amount of training data available, the complexity of the decision boundaries and the type of classifier used.

There are mainly two types of approaches in order to reduce dimensionality. Those two types are:
* projections
* transformations

The difference between one and the other is that, while projections operate on the very space defined by the set of input samples, transformations try to modify that space in order to find a transition function that allows an adequate and separable representation of the data.
Some of the most common techniques are:

* *Principal Component Analysis (PCA)*
* *Linear Discriminant Analysis (LDA)*
* *Independent Components Analysis (ICA)*
* *Locally linear embedding (LLE)*
* *t-distributed Stochastic Neighbor Embedding (t-SNE)*
* *IsoMaps*
* *Autoencoders*



# Prerequisites for this practical
The examples in this unit will be developed in Python and it will be necessary to have some libraries installed on the system to make sure they work correctly. If you are running on Colab, ignore the following cell, which is used to configure the environment on Windows.

In [ ]:
# Terminal does not work
# Type in a shell: SET SHELL=C:\Windows\SysWOW64\WindowsPowerShell\v1.0\powershell.exe

# Install Kernel in Jupyter
# conda env list
# jupyter kernelspec list
# jupyter kernelspec uninstall unwanted-kernel
# python -m ipykernel install --user --name=firstEnv

# JupyterLab start-up directory
# JupyterLab >= 3, Jupyter Notebook Classic, and RetroLab
# Open cmd (or Anaconda Prompt) and run jupyter server --generate-config instead
# This writes a file to C:\Users\username\.jupyter\jupyter_notebook_config.py.
# Browse to the file location and open it in an Editor
# Search for the following line in the file: #c.NotebookApp.notebook_dir = ''
# Replace by c.ServerApp.root_dir = '/the/path/to/home/folder/'
# Make sure you use forward slashes in your path and use /home/user/ instead of ~/ for your home directory, backslashes could be used if placed in double quotes even if folder name contains spaces as such : "D:\yourUserName\Any Folder\More Folders\"
# Remove the # at the beginning of the line to allow the line to execute

The following cell only needs to be executed once, both in Colab and locally. For all executions.

In [ ]:
# Sklearn library
!pip install scikit-learn
# matplotlib library
!pip install matplotlib
# seaborn library
!pip install seaborn

# Library for working with matrix data
!pip install numpy 
# Library for using DataFrames and queries on structured data
!pip install pandas
# Library for scientific computations with many utility functions
!pip install scipy   
# Library that makes working with files easier
!pip install pathlib 

## Loading the data for the examples
Before going any further, an example data set will be loaded so that we can see the influence of the different techniques we will address in this tutorial. The example problem that is going to be used is a classic problem known as **rock or mine?**. It is a small database consisting of 111 rock patterns and 97 patterns corresponding to underwater mines (simulated as metal cylinders). Each of the patterns consists of 60 numerical measurements between 0.0 and 1.0. Those measurements correspond to the energy value of different wavelength ranges over a certain period of time.

The first step will be to download the data set if it is not already available. To do so, the following code will be used with the utility function shown below. If, for whatever reason, you prefer to use another tool for the download, you are free to choose the one that best suits you.


In [ ]:
from pathlib import Path
import pandas as pd

def load_data(filename, url):
    # check whether the file already exists and, if not, download it
    p_filename = Path(filename)
    if not p_filename.exists():
        print(f'Downloading'.ljust(75,'.'), end='', flush=True)
        import urllib.request
        urllib.request.urlretrieve(url,p_filename)
        print(f"Done!")
    return pd.read_csv(str(p_filename), delimiter=',', header=None)

file_name = 'sonar.all_data'
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/undocumented/connectionist-bench/sonar/sonar.all-data'
data = load_data(file_name, url)

data.head(5)

### Data preprocessing
Once the data is downloaded, although the measurements are already normalized, a small adaptation of them is necessary. Specifically, it is necessary to change the interpretation of the last column, which contains the label of the problem, to an integer. In addition, the data set will also be divided into train and test. This division will keep a 10% share of the patterns for testing. A fact worth highlighting is that the split is done in such a way that the proportions of the different output classes are maintained, so it is expected that the test set has 11 rocks and 10 mines in its composition.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

# Collect the first 60 measurements and turn them into a NumPy array
# they have no names, so we access them by position
inputs = (data.iloc[:,0:60]).to_numpy()
# The last column marks the type of pattern
outputs = (data[60]=='M').astype('int')
print(f"Pattern types: {np.unique(outputs)}")

# Create the train and test sets
train_inputs, test_inputs, train_outputs, test_outputs = train_test_split(inputs, outputs, test_size=0.1, 
                                                                          stratify=outputs)

print(f"Train Patterns{train_inputs.shape} -> {train_outputs.shape}")
print(f"Test Patterns{test_inputs.shape} -> {test_outputs.shape}")

## Principal Component Analysis (PCA)

Probably the most widely used dimensionality reduction technique. It can be used both individually and in combination with other techniques. It is a method that transforms the data by projecting it onto a set of orthogonal axes. To do so, the method looks for the best linear combinations of the original variables, trying to maximize the variance along the new variable. For example, the figure on the right shows a set of points in three dimensions. When those three dimensions are projected, the images on the right show the variability of the data for each of the axes. The continuous line is the one with the greatest variability and, therefore, it will be taken as the basis or first component. For the second component, among the remaining possibilities, the one that maximizes the variance and is still perpendicular (orthogonal) to the first selected dimension will be chosen.

![](./Images/PCA.png)

If a third dimension were needed, PCA would have to look for one perpendicular to these. This process is based on the so-called *Single Value Decomposition (SVD)* matrix that extracts the eigenvectors of the sample space. They are ordered in decreasing order and the ones that best represent the corresponding space are selected.

For those who wish to understand in detail how it works, in the following [link](https://sebastianraschka.com/Articles/2014_pca_step_by_step.html) you can find a description of how to implement PCA step by step.

In general terms, if you want to make use of PCA, a good alternative is to use an implementation such as the one found in the `scikit-learn` library. That library includes the `PCA` function which allows us to run this reduction technique. See the following example:



In [ ]:
from sklearn.decomposition import PCA

#Define PCA according to what you want to keep

pca = PCA(2)

#Fit the matrices according to the training inputs
pca.fit(train_inputs)

#Once the transformation is available, it is only necessary to apply the
#transformation to the data sets

pca_train_inputs = pca.transform(train_inputs)
pca_test_inputs = pca.transform(test_inputs)

print(f"Train Patterns{train_inputs.shape} -> {pca_train_inputs.shape}")
print(f"Test Patterns{test_inputs.shape} -> {pca_test_inputs.shape}")


**Note that it is important that the fit of the transformation is done on the training data only**. If it were done over all the data, the possible trainings of classification or regression techniques that could be applied later would be contaminated by the transformation.

One of the main advantages of applying dimensionality reduction is that it allows a first visual study by transforming a multidimensional space into a 2D or 3D one that can actually be plotted. In the example, the data transformed by PCA will be used for the plot. First, a function is defined to make the presentation of the data easier:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import seaborn as sns
sns.set_style('darkgrid')
sns.set_palette('muted')
sns.set_context("notebook", font_scale=1.5,
                rc={"lines.linewidth": 2.5})

def draw_results(x, colors,target_names=None):
    """ 
        Utility function that prints a scatter plot
        in order to see how the clusters are spread
    """
    import matplotlib.patheffects as PathEffects
    
    # Select the colors (which correspond
    # to the output vector) according
    # to the number of classes. This will be the output vector.
    num_classes = len(np.unique(colors))
    palette = np.array(sns.color_palette("hls", num_classes))
    
    if target_names is not None:
        assert num_classes == len(target_names)
        label = target_names
    else:
        label = [str(i) for i in range(num_classes)]

    # Create the scatter plot
    f = plt.figure(figsize=(8, 8))
    ax = plt.subplot(aspect='equal')
    # Take only the first two dimensions of each point
    sc = ax.scatter(x[:,0], x[:,1], lw=0, s=40, 
                    c=palette[colors.astype(int)], alpha=.8)
    plt.xlim(-25, 25)
    plt.ylim(-25, 25)
    #ax.axis('off')
    ax.axis('tight')

    # Add the labels to the list of graphic elements
    txts = []

    for i in range(num_classes):
        # Place the labels at the mean values of the cluster
        xtext, ytext = np.median(x[colors == i, :], axis=0)
        txt = ax.text(xtext, ytext, label[i], fontsize=24)
        txt.set_path_effects([
            PathEffects.Stroke(linewidth=5, foreground="w"),
            PathEffects.Normal()])
        txts.append(txt)

    return f, ax, sc, txts

Had we not reduced the dimensionality, the expert would be the one responsible for choosing the two variables to plot. By doing this, one would run the risk of not representing the distribution correctly.
In the following, plot two of the dimensions you prefer and compare the results with the one obtained by PCA

In [ ]:
# Draw the PCA dataset
draw_results(pca_train_inputs, train_outputs, target_names=train_outputs.unique())

Reducing to 2 or 3 dimensions can be helpful when trying to do a first analysis, for example, to determine whether a linear classifier can give good results or whether some pattern is observed in the distribution of the data. However, the most common thing is to try to reduce the dimensionality while keeping as much variability as possible. To do so, the `scikit-learn` function allows passing a value between 0 and 1, which determines the percentage of variability that must be kept. A typical value is 0.95 since it keeps almost all the relevant information while removing a large part of the noise that could be present.

In [ ]:
# Next, carry out that reduction to 95%   
pca = PCA(0.95)
pca.fit(train_inputs)
pca_train_inputs = pca.transform(train_inputs)
pca_test_inputs = pca.transform(test_inputs)

# Compare the sizes with the ones we had before
print(f"Train Patterns{train_inputs.shape} -> {pca_train_inputs.shape}")
print(f"Test Patterns{test_inputs.shape} -> {pca_test_inputs.shape}")

In addition to being able to represent the information, reducing the dimensionality is usually associated with faster training. This is because the computational complexity and the computational effort of most learning algorithms are conditioned by the number of variables. Besides, there is also often an improvement in the models since part of the noise is removed.

In [ ]:
%%timeit -n 10
# Next, let us look at a few basic approaches and the time they take
from sklearn import svm
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import GaussianNB 

clfs = { 'SVM': svm.SVC(probability=True), 
         'LR': LogisticRegression(),
         'DT': DecisionTreeClassifier(max_depth=4),
         'NB':GaussianNB()}
base_models = ['SVM', 'LR','DT','NB']

for key in clfs.keys():
    clfs[key].fit(train_inputs, train_outputs)
    acc = clfs[key].score(test_inputs, test_outputs)
    print(f"{key}: {(acc*100):.4f}%")


Compare the results after having applied PCA to the data

In [ ]:
%%timeit -n 10
# carry out the same training as above but with the PCA data
#TODO

Another of the important parameters inside the `PCA` function indicates how the SVD matrix is computed. The implementation available in `scikit-learn` offers different alternatives, two of which stand out:
1. the first one extracts the eigenvectors based on the `LAPACK` implementation of another package known as `scipy`.
1. If the dimensions of the matrix are small (less than 500x500) and the number of components to extract is less than 80%, the implementation of the `randomized truncated SVD` algorithm presented in the articles [1, 2] is recommended

Additionally, it is worth mentioning that `scikit-learn` offers some of the most frequent variants of PCA, such as:
* [Kernel Principal Component Analysis](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.KernelPCA.html#sklearn.decomposition.KernelPCA), this method transforms the initial space by applying a kernel in which it is easier to subsequently apply PCA. Unlike PCA, it is not a linear transformation.

* [Sparse Principal Component Analysis](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.SparsePCA.html#sklearn.decomposition.SparsePCA), implementation that, firstly, looks for a set of sparse features that allows reconstructing the original set while controlling the degree of diffusion and, then, applies PCA.

* [Dimensionality reduction using truncated SVD](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html#sklearn.decomposition.TruncatedSVD), version that does not center the data before computing the SVD and can work with sparse matrices efficiently.

* [Incremental Principal Components Analysis (IPCA)](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.IncrementalPCA.html#sklearn.decomposition.IncrementalPCA) Application of PCA in batches without scaling the data before the SVD computation. It can mean an improvement in terms of memory usage, although due to the use of batches it may not be as accurate on some occasions.

[1]  [Halko, N., Martinsson, P. G., and Tropp, J. A. (2011). “Finding structure with randomness: Probabilistic algorithms for constructing approximate matrix decompositions”. SIAM review, 53(2), 217-288](https://doi.org/10.1137/090771806)

[2] [ Martinsson, P. G., Rokhlin, V., and Tygert, M. (2011). “A randomized algorithm for the decomposition of matrices”. Applied and Computational Harmonic Analysis, 30(1), 47-68](https://doi.org/10.1016/j.acha.2010.02.003)


In [ ]:
# Apply some of the variants to the problem data and plot the result, what differences can be appreciated?.
# NOTE: be careful with the random seed if a "randomized" decision is used
#TODO

## Independent Component Analysis (ICA)

Although it is often not regarded as a dimensionality reduction technique but as one for extracting the independent components, it is probably the second most popular technique applied in this sense. ICA is also a linear dimensionality reduction method, which transforms the data set into columns of independent components. Blind source separation and the "cocktail party problem" are other names for it. ICA is an important tool in the analysis of neuroimages, fMRI and EEG, helping to separate normal from abnormal signals.

In the case of this algorithm, it assumes that the presented data are the result of a linear combination of two inputs and that none of them has a Gaussian distribution. If this condition does not hold, the results will not be good or will be inconsistent.

Without going too deep into the mathematical part it is based on, one should keep in mind that the problems derived from non-linear dependencies or from having a Gaussian distribution can be minimized with the use of entropy in the formulation of the algorithm.
In general terms, the pseudo-code of ICA can be summarized as:
```
Initialize W
X = PCA(X)
While W changes:
      W = average(X*G(WX)) - average(g(WTX))W
      W = orthogonalize(W)
return S = WX
```

Where $W$ is the weight matrix that allows the change in the original data, $G$ is a negentropy matrix (computation of the entropy difference between two elements) and $g$ is the derivative of the previous function. The `orthogonalize` function refers to the process by which the columns of a matrix become orthogonal. Fortunately, a much faster implementation of this process can be found in `scikit-learn` with the [FastICA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.FastICA.html) function. Next we will see how to apply it to the data above:


In [ ]:
from sklearn.decomposition import FastICA

#Define the function and fit it
ica = FastICA(n_components=2)
ica.fit(train_inputs)

# Proceed with the transformations

ica_train_inputs = ica.transform(train_inputs)
ica_test_inputs = ica.transform(test_inputs)

print(f"Train Patterns{train_inputs.shape} -> {ica_train_inputs.shape}")
print(f"Test Patterns{test_inputs.shape} -> {ica_test_inputs.shape}")


In [ ]:
# Plot the data below and observe the differences with PCA
#TODO

# Apply a different number of components and compare the classification results with the previous techniques
#TODO

## Linear Discriminant Analysis (LDA)

This linear machine learning algorithm is used for multiclass classification, although it is also sometimes used as a dimensionality reduction algorithm. Be careful not to confuse it with "Latent Dirichlet Allocation" (LDA), which is also a dimensionality reduction technique but one that can only be applied to text documents.

LDA tries to separate (or discriminate) as well as possible the samples of the training data set by their class value. Specifically, the model tries to find a linear combination of the input variables that achieves the maximum separation of the samples between classes (centroids or class means) and the minimum separation of the samples within each class. Therefore, its main difference with PCA is that LDA takes the output class into account, while PCA is completely agnostic to this fact.

Within the `scikit-learn` library, the implementation of the algorithm allows us to use it either to classify or to transform the corresponding data. See the following example:


In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
# Take into account that the number of components when reducing dimensionality must be
# n_components <= min(n_classes - 1, n_features))  
lda = LinearDiscriminantAnalysis(n_components=1)
lda.fit(train_inputs, train_outputs)

lda_train_inputs = lda.transform(train_inputs)
lda_test_inputs = lda.transform(test_inputs)

print(f"Train Patterns{train_inputs.shape} -> {lda_train_inputs.shape}")
print(f"Test Patterns{test_inputs.shape} -> {lda_test_inputs.shape}")


Among the most important parameters of the function, the following can be highlighted:

* `solver{‘svd’, ‘lsqr’, ‘eigen’}, default=’svd’` Indicates which algorithm is used to solve the relationship
 - ‘svd’: Singular value decomposition which does not compute the covariance matrix and, therefore, is the indicated one for a large number of features.

 - ‘lsqr’: Least squares solution. Method that can use another of the parameters called *shrinkage* or custom covariance computations.

 - ‘eigen’: Eigenvalue decomposition. As with the previous method, in this case the eigenvalues of the matrix are computed.
* `n_components`, refers to the number of components that will be used for the dimensionality reduction when the `transform` method is applied. Keep in mind the general constraint $n\_components <= min(n\_classes - 1, n\_features))$

* `store_covariance (default=False)` Marks whether the class-weighted covariance matrix will be computed for the ‘svd’ estimator. In the other cases it is always computed regardless of this parameter

* `tol(default=1.0e-4)` Marks the tolerance limit for a feature to be considered significant

For more detail, please review the documentation on the [scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.discriminant_analysis.LinearDiscriminantAnalysis.html) page


In [ ]:
# Load another problem with at least 3 classes, reduce the dimensionality and plot it
#TODO


## t-Distributed Stochastic Neighbor Embedding (t-SNE)

It is one of the NON-linear techniques focused on dimensionality reduction. It is usually reserved for high-dimensionality problems. While PCA is a linear technique that seeks to maximize the variance while preserving the large distances between pairs (in other words, things that are different end up very far apart), t-SNE is based on statistics and tries to overcome the flaws of other techniques when exploring data with a non-linear structure.
![Distribution comparison](./Images/t-SNE.png) 

The algorithm starts by computing the similarity probability of the points in the high-dimensional space (Gaussian function) and computing the similarity probability of the points in the low-dimensional space (based on a Cauchy function). The similarity is computed as the conditional probability that a point $A$ chooses point $B$ as its neighbor if the neighbors are chosen in proportion to their probability density under a Gaussian (normal distribution) centered at $A$. Then, the difference between these conditional probabilities in the higher and lower dimensional space is minimized for a representation of the data points in the lower dimensional space. To measure the minimization of the sum of the conditional probability difference, t-SNE minimizes the sum of the **Kullback-Leibler divergence** of the global data points using a gradient descent method. This asymmetric statistical function, also used in *Generative Adversarial Networks (GAN)* or in *Variational Auto Encoders (VAE)*, allows optimizing the distance between the probability distributions.

In general terms, t-SNE tries to minimize the divergence between two distributions: the first one, a distribution that measures the pairwise similarities of the input objects and, the second one, a distribution that measures the pairwise similarities of the corresponding points in the low-dimensional space.

A point worth highlighting is that, as a simplification of the Cauchy distribution, it is usually simplified to a single degree of freedom, which results in a Student-t distribution, hence the name of the transformation.

The other point worth highlighting is that, after this process, the input features are no longer identifiable, and no inference can be made based solely on the t-SNE result. Hence, it is mainly an exploratory and data visualization technique.


To use it, the corresponding `scikit-learn` library can be used.


In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2)
tsne_train_inputs = tsne.fit_transform(train_inputs) 
tsne_test_inputs = tsne.fit_transform(test_inputs)

print(f"Train Patterns{train_inputs.shape} -> {tsne_train_inputs.shape}")
print(f"Test Patterns{test_inputs.shape} -> {tsne_test_inputs.shape}")


One of the elements that stand out from the previous code is that, since the t-SNE transformation cannot be stored because it is unsupervised, that transformation cannot be applied later to another data set. Additionally, since it has no meaning other than graphical and it cannot/should not be used as input to a training method, it does not have a `transform` function call.

It is worth noting that, regarding the use of the t-SNE implementation, for high-dimensional spaces (>50) or sparse spaces it is recommended to previously apply another dimensionality reduction technique such as, for example, PCA. Then, t-SNE would be applied over the result of that dimensionality reduction technique.

Among the most important parameters that can be passed to t-SNE, the following stand out:

* `n_components (default: 2)`: Dimensions to which the result is to be compressed.
* `perplexity (default: 30)`: this parameter is related to the neighborhood used by the partitions of the algorithm. The usual values are in the interval $[5,50]$
* `early_exaggeration (default: 12.0)`: Controls how tight the classes are in the reduced space and, therefore, the expected gap between them.
* `learning_rate (default: 200.0)`: as its name indicates, it is the training parameter that controls learning by gradient descent. In general terms, a value in the range $[10.0, 1000.0]$ is usually used.
* `n_iter (default: 1000)`: Maximum number of iterations applied to the optimization algorithm. In the documentation of the library it is recommended to set a number equal to or greater than 250.
* `method (default: ‘barnes_hut’)`: The method to be applied. By default, the Barnes-Hut approximation is used since it has a computational cost of $O(N log(N))$. Alternatively, although slower since it has a complexity of $O(N^2)$, the ’exact’ method can be used which, as its name indicates, makes no assumptions or approximations but computes the exact value.


In [ ]:
# Print the representation of the patterns
#TODO



## Exercise

Load the [fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist) problem and apply the *PCA*, *LDA* and *t-SNE* techniques. Analyze/Plot the differences in the result graphically. You can use the `draw_results` function defined earlier in this tutorial to easily represent the 10 classes of this classification problem.

The problem in question has 70,000 images of 10 different types of clothing. Each class is represented by 7,000 images. The classes considered, as described in the problem document, are:

0. T-shirt/top
1. Trouser
2. Pullover
3. Dress
4. Coat
5. Sandal
6. Shirt
7. Sneaker
8. Bag
9. Ankle boot

The patterns are images of $28\times28$ pixels with the image centered. These patterns have been split following a *hold-out* scheme with 60,000 images for training and 10,000 for testing.

In this case, we are only going to use the training images and, depending on the capabilities of the system, it may be necessary to reduce the number of those to which the reduction techniques will be applied so that it can be done in an acceptable time.


In [ ]:
def load_mnist(path_str, pattern_sz=(28,28), kind='train'):
    from pathlib import Path
    import gzip
    import numpy as np

    """Load MNIST-type problems with images and labels"""
    path = Path(path_str)
    labels_path = path / f'{kind}-labels-idx1-ubyte.gz'
    images_path = path / f'{kind}-images-idx3-ubyte.gz'

    with gzip.open(str(labels_path.absolute()), 'rb') as lbpath:
        labels = np.frombuffer(lbpath.read(), dtype=np.uint8,
                               offset=8)
        
    with gzip.open(str(images_path.absolute()), 'rb') as imgpath:
        images = np.frombuffer(imgpath.read(), dtype=np.uint8,
                               offset=16).reshape(len(labels), pattern_sz[0]*pattern_sz[1])
        
    return images, labels

#TODO
download_path = 'Adjust according to where the files have been downloaded'

X_train, y_train = load_mnist(download_path, kind='train')

#If the machine capabilities are limited, reduce the number of patterns from 60,000 to 20,000

#TODO if necessary

#Check that you have the 10 classes in the selected subsample; otherwise shuffle 
#and select a stratified set
#TODO

#carry out the reduction with PCA, LDA and t-SNE 
#TODO

#Plot the results
#TODO